# Security audit of the dataset encryption engine

Dataset Generation, IT22541284 (Yubitha S.)

This notebook runs `scripts/security_audit.py` on the real `SecurityEngine` and shows the charts.
It does not copy the engine code, so it always tests what the capture uses.

| Test | What it checks |
|---|---|
| Round trip | Encrypt then decrypt returns the exact original bytes |
| Nonce uniqueness | The same file encrypted 30 times gives 30 different IVs or nonces |
| Byte distribution | Ciphertext bytes look uniform (chi-square, 255 degrees of freedom, critical value 293.25) |
| Ciphertext change | A 1-bit change in the file changes about half the ciphertext bits |
| Tamper detection | A changed ciphertext byte is rejected (GCM only; CBC has no tag) |
| Real folder | A captured town holds no readable PNG; with `VERIFY = True`, every file decrypts and matches the manifest |

Tests 1 to 5 use synthetic bytes and a random key that exists only in memory.
The real-folder check reads the dataset key from the project `.env`. No key is written in this notebook.

Setup, from the `Dataset-Generation` folder: `pip install -e ".[audit]"`

In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import Image, display

HERE = Path.cwd()
SCRIPTS = (HERE.parent if HERE.name == "notebooks" else HERE) / "scripts"
sys.path.insert(0, str(SCRIPTS))

import security_audit  # noqa: E402

MODE = "gcm"        # "gcm" or "cbc"
DATA_DIR = None     # for example r"..\output\town01"; None skips the real-folder test
VERIFY = False      # True: decrypt every file in memory and compare with the manifest
OUT = Path("outputs") / "security_audit"

In [ ]:
report = security_audit.run_audit(MODE, DATA_DIR, VERIFY)
for name, result in report["tests"].items():
    status = "N/A" if result["passed"] is None else ("PASS" if result["passed"] else "FAIL")
    print(f"{status:5} {name}")
print(f"\n{report['cipher']}: {report['passed']} / {report['decided']} passed, verdict {report['verdict']}")

In [ ]:
shown = {k: {f: v for f, v in t.items() if f != "counts"} for k, t in report["tests"].items()}
print(json.dumps(shown, indent=2))

In [ ]:
os.makedirs(OUT, exist_ok=True)
for path in security_audit.save_charts(report, str(OUT)):
    display(Image(filename=path))

Command-line equivalent, which also saves `security_audit.json`:

```
python scripts/security_audit.py --mode gcm --data-dir output/town01 --verify
```